# Lab 03 — Messy Retail Cleanup Pipeline
**Data Quality Track** · Beginner–Intermediate · ~60 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Profile a dirty 1000-row inventory export before cleaning
2. Parse word-numbers ('two hundred') and reject NaN prices
3. Validate Status against an allowed set with a flag column
4. Compute stock value by Category and export warehouse_clean.csv

## Datasets (this folder)
- `warehouse_messy_data.csv` — auto-download from `https://raw.githubusercontent.com/eyowhite/Messy-dataset/main/warehouse_messy_data.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-01-foundations-python-math-data/labs/lab-03-messy-retail-cleanup/lab-03-messy-retail-cleanup.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0** first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 1).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`warehouse_messy_data.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-03-messy-retail-cleanup"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-01-foundations-python-math-data/bundles/lab-03-messy-retail-cleanup/dataset.zip"
NEED = ["warehouse_messy_data.csv"]  # unzipped files used by the code below

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Data Quality Track: From Dirty Warehouse Export to Clean CSV

> **Scenario:** Warehouse hands you `warehouse_messy_data.csv` — 1000 rows of product inventory with mixed-case names, `Quantity` as `"two hundred"` / `"NaN"`, missing prices, inconsistent `Status`. Clean it with a reproducible pipeline and report **stock value by Category**.
>
> **You will learn:** string normalisation, numeric coercion with fallbacks, date parsing (`dd/mm/yyyy`), validation rules, writing a cleaned CSV.
> **Time:** ~60 minutes. **Level:** Beginner–Intermediate. **Needs:** Python 3.8+ (pandas optional). **Env:** 🟢 Colab only.

### Dirty-data mental map

| Mess in the file | Clean rule | Python tool |
|---|---|---|
| `" gadget y "` / `"WIDGET A"` | strip + title-case | `.strip().title()` |
| `"two hundred"` / `"NaN"` qty | word-number map or `None` | custom `parse_qty` |
| `"NaN"` price | treat as missing | `parse_price` → `None` |
| `"Low Stock"` status | flag outside allowed set | set membership |
| `"20/12/2022"` restock date | parse `dd/mm/yyyy` | `datetime.strptime` |

---

### 1. Load and profile (local first, Colab fallback)

In [ ]:
import csv, os, math, re
from datetime import datetime
from collections import defaultdict, Counter

def load_rows():
    local = "warehouse_messy_data.csv"
    if not os.path.exists(local):
        import urllib.request
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/eyowhite/Messy-dataset/main/warehouse_messy_data.csv",
            local,
        )
    with open(local, newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

rows = load_rows()
print(len(rows), "rows")
print("columns:", list(rows[0].keys()))
# 1000 rows
# columns: ['Product ID', 'Product Name', 'Category', 'Warehouse', 'Location',
#           'Quantity', 'Price', 'Supplier', 'Status', 'Last Restocked']

print("Status values:", Counter(r["Status"] for r in rows))
# {'In Stock': 340, 'Out of Stock': 332, 'Low Stock': 328}

print("Quantity samples:", Counter(r["Quantity"] for r in rows).most_common(6))
# includes 'two hundred': 160, 'NaN': 158


> Profile first: value counts on categorical fields catch dirty levels (`Low Stock`) before you write cleaning code.

---

### 2. Clean text columns

In [ ]:
def clean_text(value):
    if value is None:
        return ""
    s = str(value).strip()
    if s.lower() in ("nan", "none", ""):
        return ""
    return s.title()  # ' gadget y ' -> 'Gadget Y', 'ELECTRONICS' -> 'Electronics'

for r in rows[:3]:
    print(repr(r["Product Name"]), "->", clean_text(r["Product Name"]),
          "|", r["Category"], "->", clean_text(r["Category"]))
# ' gadget y ' -> 'Gadget Y' | ELECTRONICS -> Electronics


---

### 3. Parse Quantity (word-numbers + missing)

In [ ]:
WORDMAP = {
    "zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
    "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
    "twenty": 20, "thirty": 30, "forty": 40, "fifty": 50,
    "hundred": 100, "thousand": 1000,
}

def parse_qty(q):
    """Return int quantity or None. Handles '300', 'two hundred', 'NaN'."""
    q = str(q).strip().lower()
    if q in ("", "nan", "none"):
        return None
    try:
        return int(float(q))
    except ValueError:
        total = 0
        tokens = q.replace("-", " ").replace("and", " ").split()
        if "hundred" in tokens:
            i = tokens.index("hundred")
            total += (WORDMAP.get(tokens[i - 1], 1) if i > 0 else 1) * 100
            tokens = tokens[i + 1:]
        if "thousand" in tokens:
            i = tokens.index("thousand")
            total += (WORDMAP.get(tokens[i - 1], 1) if i > 0 else 1) * 1000
            tokens = tokens[i + 1:]
        for t in tokens:
            total += WORDMAP.get(t, 0)
        return total if total else None

print(parse_qty("300"), parse_qty("two hundred"), parse_qty("NaN"))
# 300 200 None

qty_ok = sum(1 for r in rows if parse_qty(r["Quantity"]) is not None)
print("qty parseable:", qty_ok)  # 842 of 1000 (158 are NaN)


---

### 4. Parse Price (reject NaN) and dates

In [ ]:
def parse_price(p):
    """Float price or None. Rejects NaN/empty."""
    s = str(p).strip().replace("$", "").replace(",", "")
    if s.lower() in ("", "nan", "none"):
        return None
    try:
        v = float(s)
        return None if math.isnan(v) else v
    except ValueError:
        return None

def parse_date(s):
    """dd/mm/yyyy or ISO; None if missing."""
    s = str(s).strip()
    if s.lower() in ("nan", "none", ""):
        return None
    for fmt in ("%d/%m/%Y", "%Y-%m-%d", "%m/%d/%Y"):
        try:
            return datetime.strptime(s, fmt).date()
        except ValueError:
            continue
    return None

print(parse_price("19.99"), parse_price("NaN"))   # 19.99 None
print(parse_date("20/12/2022"), parse_date("NaN"))  # 2022-12-20 None

price_ok = sum(1 for r in rows if parse_price(r["Price"]) is not None)
date_ok  = sum(1 for r in rows if parse_date(r["Last Restocked"]) is not None)
print("price ok:", price_ok, "date ok:", date_ok)  # 793 price, 800 date


---

### 5. Validate Status and build cleaned rows

In [ ]:
ALLOWED_STATUS = {"In Stock", "Out of Stock"}

def clean_row(r):
    qty = parse_qty(r["Quantity"])
    price = parse_price(r["Price"])
    status_raw = str(r["Status"]).strip()
    return {
        "Product ID": str(r["Product ID"]).strip(),
        "Product Name": clean_text(r["Product Name"]),
        "Category": clean_text(r["Category"]),
        "Warehouse": clean_text(r["Warehouse"]),
        "Location": clean_text(r["Location"]),
        "Quantity": qty,
        "Price": price,
        "Supplier": clean_text(r["Supplier"]),
        "Status": status_raw,
        "Status OK": status_raw in ALLOWED_STATUS,
        "Last Restocked": parse_date(r["Last Restocked"]),
        "Stock Value": round(qty * price, 2) if (qty and price) else None,
    }

cleaned = [clean_row(r) for r in rows]
bad_status = [c for c in cleaned if not c["Status OK"]]
print("invalid status rows:", len(bad_status), "->", Counter(c["Status"] for c in bad_status))
# 328 -> {'Low Stock': 328}
print("sample cleaned:", cleaned[0])


> `Status OK` is a **flag column**, not a silent drop — auditors need to see what you excluded.

---

### 6. Stock value by Category + write clean CSV

In [ ]:
by_cat = defaultdict(lambda: {"rows": 0, "value": 0.0, "units": 0})
for c in cleaned:
    cat = c["Category"] or "Unknown"
    by_cat[cat]["rows"] += 1
    if c["Stock Value"]:
        by_cat[cat]["value"] += c["Stock Value"]
        by_cat[cat]["units"] += c["Quantity"] or 0

for cat in sorted(by_cat):
    b = by_cat[cat]
    print(f"{cat:12s}  rows={b['rows']:4d}  units={b['units']:6d}  value=${b['value']:,.2f}")
# Electronics  rows= 248  ... value=$830,214.00
# Clothing     rows= 257  ... value=$716,252.50
# Furniture    rows= 265  ... value=$796,211.50
# Toys         rows= 230  ... value=$607,261.00
print("TOTAL value:", round(sum(b["value"] for b in by_cat.values()), 2))
# TOTAL value: 2949939.0  (only rows with both qty and price: 661)

# Persist clean file next to the notebook
out_path = "warehouse_clean.csv"
fields = ["Product ID", "Product Name", "Category", "Warehouse", "Location",
          "Quantity", "Price", "Supplier", "Status", "Status OK",
          "Last Restocked", "Stock Value"]
with open(out_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
    w.writeheader()
    for c in cleaned:
        row = dict(c)
        row["Last Restocked"] = str(c["Last Restocked"] or "")
        w.writerow(row)
print("wrote", out_path, "rows:", len(cleaned))


---

## Exercises (do these!)

### Exercise 1 — Fix Quantity word-numbers
Convert every `Quantity` with `parse_qty`. Print how many rows were `"two hundred"`, how many `"NaN"`, and the sum of all parseable quantities.
*Expected: 160 × "two hundred" → 200 each; 158 × NaN → None; 842 parseable rows; sum of parseable qty = 135,900 (run the sum yourself to confirm).*

**Follow-up:** What fraction of rows have parseable quantities? Check: 0.842.

<details>
<summary>Hint</summary>

```python
from collections import Counter
c = Counter(r["Quantity"] for r in rows)
words = c["two hundred"]; missing = c["NaN"]
total = sum(parse_qty(r["Quantity"]) or 0 for r in rows)
```

</details>

### Exercise 2 — Flag invalid Status
Using `ALLOWED_STATUS = {"In Stock", "Out of Stock"}`, count rows whose `Status` is **not** allowed. Which status value causes all of them?
*Expected: 328 rows, all `"Low Stock"`.*

**Follow-up:** What is the average stock value per usable row? Check: about 4462.84.

<details>
<summary>Hint</summary>

```python
bad = [r for r in rows if str(r["Status"]).strip() not in ALLOWED_STATUS]
print(len(bad), Counter(r["Status"] for r in bad))
```

</details>

### Exercise 3 — Inventory value per Category
Compute `Stock Value = Quantity * Price` only when both parse; sum by `Category`. Print the four category totals and the grand total.
*Expected (2 d.p.): Electronics 830214.00 · Clothing 716252.50 · Furniture 796211.50 · Toys 607261.00 · Grand total 2949939.00 (661 usable rows).*

**Follow-up:** Which category holds the most value? Check: Electronics.

<details>
<summary>Hint</summary>

Skip rows where `parse_qty` or `parse_price` returns `None` — do **not** treat missing as 0 for value (that understates inventory).
</details>

---

## Solutions

Try for 15 min each before peeking.

In [ ]:
# --- Solution 1 ---
from collections import Counter
c = Counter(r["Quantity"] for r in rows)
print("two hundred:", c["two hundred"], "NaN:", c["NaN"])
parseable = [parse_qty(r["Quantity"]) for r in rows]
print("parseable:", sum(1 for q in parseable if q is not None))
print("sum qty:", sum(q for q in parseable if q is not None))
# two hundred: 160 NaN: 158
# parseable: 842
# sum qty: 135900

# --- Solution 2 ---
ALLOWED = {"In Stock", "Out of Stock"}
bad = [r for r in rows if str(r["Status"]).strip() not in ALLOWED]
print(len(bad), Counter(str(r["Status"]).strip() for r in bad))
# 328 Counter({'Low Stock': 328})

# --- Solution 3 ---
by_cat = defaultdict(float)
usable = 0
for r in rows:
    q, p = parse_qty(r["Quantity"]), parse_price(r["Price"])
    if q and p:
        by_cat[clean_text(r["Category"])] += q * p
        usable += 1
for cat in sorted(by_cat):
    print(f"{cat}: {by_cat[cat]:,.2f}")
print(f"TOTAL: {sum(by_cat.values()):,.2f}  (rows={usable})")
# Clothing: 716,252.50
# Electronics: 830,214.00
# Furniture: 796,211.50
# Toys: 607,261.00
# TOTAL: 2,949,939.00  (rows=661)

# --- Follow-up 1 ---
pshr = round(sum(1 for q in parseable if q is not None) / len(rows), 4)
print(pshr)  # 0.842
assert pshr == 0.842

# --- Follow-up 2 ---
per_row = round(sum(by_cat.values()) / usable, 2)
print(per_row)  # ~4462.84
assert per_row == 4462.84

# --- Follow-up 3 ---
top = max(by_cat, key=by_cat.get)
print(top)  # Electronics
assert top == "Electronics"


### What to learn next
- pandas equivalent: `pd.to_numeric(..., errors="coerce")` + `.fillna`.
- Great Expectations / pandera for reusable column schemas.
- Then: unit tests for `parse_qty` (edge cases: `"twenty five"`, `""`, `"1e3"`).
- Cheat sheet: profile → parse → flag → aggregate → export; never drop rows silently.

*Files in this folder: `warehouse_messy_data.csv` · output `warehouse_clean.csv` (created when you run Section 6). Paste any block into Python/Jupyter and run top-to-bottom.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
